In [ ]:
%pip install transformers langchain torch langchain-community langchain-huggingface
% pip install -U "typing_extensions>=4.14.0"


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from transformers import T5Tokenizer, T5ForConditionalGeneration
from langchain_huggingface import HuggingFacePipeline

model_name = "microsoft/phi-4"

if model_name.startswith("t5"):
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)

    hf_pipeline = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=512,
        max_new_tokens=50,
        truncation=True
    )
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    tokenizer.pad_token = tokenizer.eos_token

    hf_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=512,
        truncation=True
    )

llm = HuggingFacePipeline(pipeline=hf_pipeline)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00006.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00004-of-00006.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

model-00005-of-00006.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
### Try this for gpt2 and t5-small
prompt = "How are you?"

output = llm(prompt)

output

In [ ]:
from langchain.prompts import PromptTemplate, ChatPromptTemplate

template = PromptTemplate(
    input_variables=["text"],
    template="translate English to French: {text}"
)

### Try this for gpt2 and t5-small
prompt_text = template.format(text="How are you?")

response = llm(prompt_text)

response

In [ ]:
template_string = """You be tasked with takin' the followin' text \
and transformin' it into a joke in the style of {style}. \
Make sure it stays true to the humor and tone of the given style. \
If the text ain't naturally a joke, twist it into somethin' funny! 

text: ```{text}```
"""

In [ ]:
prompt_template = PromptTemplate.from_template(template_string)

prompt_template

In [ ]:
prompt_template.input_variables

In [ ]:
prompt_style = """pirate"""

prompt_input = """Why do programmers prefer dark mode?"""

In [ ]:
message = prompt_template.format(
                    style=customer_style,
                    text=customer_email)

response = llm(message)

response

In [ ]:
from langchain.chains import LLMChain

prompt = ChatPromptTemplate.from_template(
    "Tell a joke about {topic} in the style of {style}."
)

chain = LLMChain(llm=llm, prompt=prompt)

# Example Input
topic = "why pirates love gold"
style = "pirate"

chain.invoke({"topic": topic, "style": style})

In [ ]:
## SequentialChain

from langchain.chains import SequentialChain

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )

third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)

chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )

fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )

overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

review = "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?"

overall_chain(review)

In [ ]:
## Router Chain
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser

## .... 